# ESMA: Evolution Strategies for Metacognitive Alignment
## Gemma 4 E2B LoRA Fine-tuning via M-ratio Reward Signal

**Hackathon:** Gemma 4 Good (https://www.kaggle.com/competitions/gemma-4-good-hackathon)  
**Model:** google/gemma-4-2b-it (Gemma 4 E2B instruction-tuned)  
**Method:** Gradient-free evolution strategy over LoRA adapter weights, optimising the M-ratio (meta-d′/d′) reward signal

## Why metacognitive alignment?

A model aligned to maximise M-ratio has learned to *know what it knows*. This is a direct 
benefit for social good applications: a medical assistant that can reliably distinguish its 
confident recommendations from its uncertain guesses is safer to deploy than one that is
uniformly overconfident. ESMA fine-tunes metacognition directly, without requiring any 
labelled metacognitive training data.

## Algorithm

```
For each epoch:
  1. Sample 8 Gaussian perturbations of the LoRA adapters
  2. Evaluate each perturbed model on a 30-item held-out batch
     (mini two-turn confidence elicitation → M-ratio reward)
  3. Softmax-weighted average update toward high-M-ratio region
  4. Log M-ratio trajectory for longitudinal tracking
```

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.27.0 transformers>=4.40.0

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/mcbench-metacognitive-benchmark')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Run ESMA on Gemma 4 E2B ───────────────────────────────────────────────────
from src.esma.esma import run_esma

results = run_esma(
    model_name='google/gemma-4-2b-it',
    n_epochs=8,
    population_size=8,
    batch_size=30,
    noise_std=0.01,
)

In [ ]:
# ── Plot M-ratio trajectory ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

trajectory = results['m_ratio_trajectory']
epochs = list(range(len(trajectory)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('ESMA: Metacognitive Alignment of Gemma 4 E2B via M-ratio Optimisation', fontsize=12)

# M-ratio over epochs
axes[0].plot(epochs, trajectory, 'o-', color='#e74c3c', linewidth=2, markersize=6)
axes[0].axhline(trajectory[0], color='gray', linestyle='--', alpha=0.6, label=f'Baseline={trajectory[0]:.3f}')
axes[0].axhline(1.0, color='green', linestyle='--', alpha=0.4, label='Ideal M=1')
axes[0].fill_between(epochs, trajectory[0], trajectory, alpha=0.15, color='#e74c3c')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('M-ratio (meta-d\'/d\')')
axes[0].set_title('M-ratio Trajectory')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Population reward distribution per epoch
if results.get('all_rewards'):
    ax2 = axes[1]
    for i, rewards in enumerate(results['all_rewards'], start=1):
        ax2.scatter([i] * len(rewards), rewards, alpha=0.5, color='steelblue', s=20)
    ax2.plot(range(1, len(results['all_rewards'])+1),
             [np.max(r) for r in results['all_rewards']],
             'r-', linewidth=1.5, label='Best variant')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Population M-ratio')
    ax2.set_title('NES Population Rewards')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('esma_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: esma_trajectory.png')

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
baseline = results['baseline_m_ratio']
best = results['best_m_ratio']
improvement = best - baseline
best_epoch = results['best_epoch']

print('=' * 50)
print('ESMA RESULTS: Gemma 4 E2B Metacognitive Alignment')
print('=' * 50)
print(f'Baseline M-ratio (pre-ESMA):  {baseline:.4f}')
print(f'Best M-ratio (epoch {best_epoch}):     {best:.4f}')
print(f'Improvement:                  +{improvement:.4f} ({improvement/max(baseline,0.001)*100:.1f}%)')
print(f'Final model saved at:         {results["final_model_path"]}')
print()
print('Trajectory:', [round(x, 4) for x in results['m_ratio_trajectory']])

monotone_epochs = sum(
    1 for i in range(1, len(results['m_ratio_trajectory']))
    if results['m_ratio_trajectory'][i] >= results['m_ratio_trajectory'][i-1]
)
print(f'Monotone increase rate: {monotone_epochs}/{len(results["m_ratio_trajectory"])-1} epochs')